# ML-04 — Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Schema expectations

*What columns does your lane need, what types, and what are the nullability rules?*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n--- Column types ---")
print(df.dtypes.value_counts())

In [ ]:
# Required columns for Lane 2 — must exist and have expected types
required_cols = {
    # Identifiers (grouping only, never features)
    "content_id": "object",   # Unique per row
    "client_id": "object",    # 32 distinct values, for client-holdout splits
    
    # Label source (never a feature)
    "trend_direction": "object",  # {down, stable, up, new, flat}
    
    # Numeric features
    "impressions_90d": "float64",  # Must be > 0 after filtering
    "clicks_90d": "float64",
    "sessions_90d": "float64",
    "content_age_days": "float64", # Must be >= 90 after filtering
    "days_since_last_update": "float64",
    "avg_position": "float64",     # 0 = no data, not position zero
    "ctr": "float64",              # ×100 percentage: 0.76 = 0.76%
    "engagement_rate": "float64",  # ×100 percentage
    "scroll_rate": "float64",      # Can exceed 100
}

for col, expected_kind in required_cols.items():
    exists = col in df.columns
    actual = str(df[col].dtype) if exists else "MISSING"
    print(f"  {'✅' if exists else '❌'} {col:30s} expected={expected_kind:10s} actual={actual}")

## 2. Nullability and missingness patterns

*Which columns have missing values and why? Is missingness random or systematic?*

In [ ]:
# Check missingness
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
missing = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
missing = missing[missing["null_count"] > 0].sort_values("null_count", ascending=False)
print(f"Columns with missing values: {len(missing)} of {df.shape[1]}")
print(missing)

In [ ]:
# Missingness is systematic, not random — check by content_type
keyword_cols = ["search_volume", "competition", "competition_level", "cpc"]
for ctype in df["content_type"].unique():
    subset = df[df["content_type"] == ctype]
    nulls = subset[keyword_cols].isnull().mean().mean() * 100
    print(f"  {ctype:25s}  keyword-column null rate: {nulls:.1f}%")

print("\n⚠ Keyword columns are missing along content_type lines — not random.")
print("  A blind fillna(0) would silently encode content_type into features.")

## 3. Columns excluded and why

*Which columns are deliberately excluded from features and the rationale for each.*

| Column | Excluded | Reason |
|---|---|---|
| `trend_direction` | ✅ Yes | **Label leakage.** `is_declining_label` is derived from this column. Including it gives the model the answer. |
| `trend_pct` | ✅ Yes | **Label leakage.** The continuous percentage that generates `trend_direction` buckets. |
| `content_id` | ✅ Yes | **Pseudonymous identifier.** For grouping/joins only. The model should learn patterns, not memorize IDs. |
| `client_id` | ✅ Yes | **Pseudonymous identifier.** Used for client-holdout splits only. |
| `provider_used` | ✅ Yes | **Not a search-performance signal.** Editorial metadata about which LLM wrote the content. |
| `model_used` | ✅ Yes | **Not a search-performance signal.** Same as provider_used — editorial context only. |
| `char_count_tier` | ✅ Yes | **Redundant with char_count.** The continuous feature is more informative than the bucketed tier. |

**Rate-column convention:** `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and `trend_pct` are all ×100 percentages — `0.76` means **0.76%**, not 76%. This is documented in the data dictionary and must be respected in any thresholding or interpretation.

## 4. The contract in code

*Programmatic checks that validate the data contract before training.*

In [ ]:
def validate_data_contract(df):
    """Run all data contract checks. Raises AssertionError on failure."""
    checks = []
    
    # 1. Required columns exist
    required = ["content_id", "client_id", "impressions_90d", "sessions_90d",
                "content_age_days", "trend_direction"]
    for col in required:
        ok = col in df.columns
        checks.append((f"Column '{col}' exists", ok))
    
    # 2. No duplicate content_ids
    checks.append(("No duplicate content_id", df["content_id"].is_unique))
    
    # 3. trend_direction has expected values
    valid_trends = {"down", "stable", "up", "new", "flat"}
    actual_trends = set(df["trend_direction"].dropna().str.lower().unique())
    checks.append(("trend_direction values valid", actual_trends.issubset(valid_trends)))
    
    # 4. Rate columns are ×100 (not 0-1 scale)
    if "ctr" in df.columns:
        checks.append(("CTR is ×100 (max < 100)", df["ctr"].max() < 100))
    
    # 5. avg_position: 0 means no data
    checks.append(("avg_position=0 rows exist (no data)", (df["avg_position"] == 0).sum() > 0))
    
    for name, passed in checks:
        print(f"  {'✅' if passed else '❌'} {name}")
    
    failures = [name for name, passed in checks if not passed]
    if failures:
        raise AssertionError(f"Data contract violations: {failures}")
    print(f"\n✅ All {len(checks)} contract checks passed.")

validate_data_contract(df)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.